# Speculative Execution | Reasoning Patterns

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict
from typing_extensions import NotRequired
from concurrent.futures import ThreadPoolExecutor
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import time

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Pre-defined intent handlers — in production these could be full agent chains
INTENT_RESPONSES = {
    "billing": "You are a billing specialist. Help the customer with their billing issue: {query}",
    "technical": "You are a technical support engineer. Troubleshoot this issue: {query}",
    "cancellation": "You are a retention specialist. Address this cancellation request empathetically: {query}",
}

class SpecState(TypedDict):
    query: str
    intent: NotRequired[str]
    speculative_cache: NotRequired[Dict[str, str]]
    final_response: NotRequired[str]
    cache_hit: NotRequired[bool]

def speculate_and_classify(state: SpecState) -> dict:
    """Run intent classification AND all speculative responses in parallel."""
    timings = {}

    def classify() -> str:
        start = time.time()
        response = model.invoke(
            f"Classify this customer message into exactly one intent: "
            f"'billing', 'technical', or 'cancellation'.\n\n"
            f"Message: {state['query']}\n\nRespond with ONLY the intent word."
        )
        timings["classify"] = time.time() - start
        return response.content.strip().lower()

    def speculate(intent: str) -> tuple:
        start = time.time()
        prompt = INTENT_RESPONSES[intent].format(query=state["query"])
        response = model.invoke(prompt)
        timings[f"spec_{intent}"] = time.time() - start
        return (intent, response.content)

    with ThreadPoolExecutor(max_workers=4) as executor:
        # Launch classification and ALL speculative branches simultaneously
        classify_future = executor.submit(classify)
        spec_futures = {intent: executor.submit(speculate, intent) for intent in INTENT_RESPONSES}

        intent = classify_future.result()
        cache = {name: f.result()[1] for name, f in spec_futures.items()}

    if intent not in INTENT_RESPONSES:
        intent = "technical"

    return {"intent": intent, "speculative_cache": cache}

def select_result(state: SpecState) -> dict:
    """Select the pre-computed response — instant because it was speculated."""
    intent = state["intent"]
    cached = state["speculative_cache"].get(intent)
    if cached:
        return {"final_response": cached, "cache_hit": True}
    # Fallback: compute on demand (should not happen if intents match)
    response = model.invoke(INTENT_RESPONSES[intent].format(query=state["query"]))
    return {"final_response": response.content, "cache_hit": False}

In [5]:
graph = StateGraph(SpecState)
graph.add_node("speculate", speculate_and_classify)
graph.add_node("select", select_result)
graph.add_edge(START, "speculate")
graph.add_edge("speculate", "select")
graph.add_edge("select", END)

spec_exec = graph.compile()

In [6]:
# Plot the workflow
plot_mermaid(spec_exec)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	speculate(speculate)
	select(select)
	__end__([<p>__end__</p>]):::last
	__start__ --> speculate;
	speculate --> select;
	select --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
result = spec_exec.invoke({
    "query": "I've been charged twice for my last invoice and I want a refund immediately."
})
print(f"Intent: {result['intent']} | Cache hit: {result['cache_hit']}")
print(f"Response: {result['final_response'][:300]}")

Intent: billing | Cache hit: True
Response: I'm sorry to hear that you've been charged twice. Let's work on resolving this issue promptly. Here’s what we can do:

1. **Verify the Charges:** First, please check your bank statements or online account for transaction details. This will help confirm that you have indeed been charged twice for the


In [8]:
stream_invoke(spec_exec, {
    "query": "I've been charged twice for my last invoice and I want a refund immediately."
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'query': "I've been charged twice for my last invoice and I want a refund immediately.",
 'intent': 'billing',
 'speculative_cache': {'billing': "I'm sorry to hear that you've been charged twice. Let's work on resolving this issue promptly. Here’s what we can do:\n\n1. **Verify the Charges:** First, please check your bank statements or online account for transaction details. This will help confirm that you have indeed been charged twice for the same invoice.\n\n2. **Obtain Invoice Details:** Could you provide me with the invoice number or date of the transaction? This will help me quickly locate your billing records.\n\n3. **Review Account Information:** I'll need your account details or customer ID to access your billing information. Please ensure you are sharing this information through a secure channel.\n\n4. **Process the Refund:** Once we verify the duplicate charge, I will initiate a refund request for the duplicate payment. Refunds typically take a few business days to process,